In [ ]:
from notebook.services.config import ConfigManager
cm = ConfigManager()
cm.update('livereveal', {'width': 1920, 'height': 1080, 'scroll': True})

# Week 14: Monday, AST 5011: Astrophysical Systems

## Chemical Evolution & Cosmic Reionization

### Michael Coughlin

References (beyond CFN):
- Part I: Matteucci, *Chemical Evolution of Galaxies* (2012); Mo, van den Bosch & White, Ch. 10
- Part II: Loeb & Furlanetto, *The First Galaxies in the Universe* (2013); Barkana & Loeb 2001 review; MBW Ch. 16

With material from Benedikt Diemer (UMD).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from colossus.cosmology import cosmology

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

cosmo = cosmology.setCosmology('planck18')

# ---- Chemical evolution helpers ----

Z_SUN = 0.0142
LOGOH_SUN = 8.69
Y_Z_DEFAULT = 0.02

def closedBoxZ(mu, yield_Z=Y_Z_DEFAULT):
    """Closed-box metallicity: Z(mu) = y_Z ln(1/mu)."""
    mu = np.clip(np.asarray(mu, dtype=float), 1e-6, 1.0)
    return yield_Z * np.log(1.0 / mu)

def closedBoxMDF(Z, Z_final, yield_Z=Y_Z_DEFAULT):
    """Cumulative MDF in the closed box (G-dwarf prediction)."""
    num = 1.0 - np.exp(-np.asarray(Z) / yield_Z)
    den = 1.0 - np.exp(-Z_final / yield_Z)
    return num / den

def accretingBoxZ(mu, yield_Z=Y_Z_DEFAULT, eta=1.0):
    """Bathtub-model approach to equilibrium Z_eq = y_Z/(1+eta)."""
    Z_eq = yield_Z / (1.0 + eta)
    mu = np.clip(np.asarray(mu, dtype=float), 1e-6, 1.0)
    return Z_eq * (1.0 - mu)

def massMetallicityRelation(log_Mstar, z=0.0):
    """12 + log10(O/H) vs log M* (Tremonti+2004 / Zahid+2014 style)."""
    log_Mstar = np.asarray(log_Mstar, dtype=float)
    log_Msat = 10.5 + 0.3 * z
    slope = 0.5 * np.exp(-0.1 * z)
    log_OH_sat = LOGOH_SUN + 0.05 - 0.1 * z
    excess = np.clip(log_Mstar - log_Msat, -2.5, 0.0)
    return log_OH_sat + slope * excess

def alphaFeVsTimescale(tau_SF_Gyr):
    """[alpha/Fe] vs SF timescale (short bursts -> enhanced alpha)."""
    tau = np.clip(np.asarray(tau_SF_Gyr, dtype=float), 0.1, 30.0)
    return 0.3 - 0.3 * np.log10(tau)

# ---- Reionization helpers ----

SIGMA_T = 6.6524e-25
C_CMS = 2.998e10
MPC_CM = 3.0857e24
M_PROTON = 1.6726e-24

def sfrdMadau14(z):
    return 0.015 * (1 + z)**2.7 / (1.0 + ((1 + z) / 2.9)**5.6)

def ionizingEmissivity(z, f_esc=0.2, log_xi_ion=25.3):
    rho_sfr = sfrdMadau14(z)
    rho_L_UV = rho_sfr / 1.15e-28
    return f_esc * 10**log_xi_ion * rho_L_UV

def hydrogenNumberDensity(z=0.0):
    rho_crit_0 = 1.878e-29 * cosmo.h**2
    return cosmo.Ob0 * rho_crit_0 * 0.75 / M_PROTON

def recombinationTimescale(z, T=2e4, C_HII=3.0):
    alpha_B = 2.6e-13 * (T / 1e4)**(-0.76)
    n_H = hydrogenNumberDensity() * (1.0 + z)**3
    return 1.0 / (C_HII * alpha_B * n_H)

def reionizationHistory(z_grid, f_esc=0.2, log_xi_ion=25.3, C_HII=3.0):
    z_arr = np.asarray(z_grid, dtype=float)
    order = np.argsort(-z_arr)
    z_sorted = z_arr[order]
    Q = np.zeros_like(z_sorted)
    n_H0 = hydrogenNumberDensity()
    t_arr = cosmo.age(z_sorted) * 3.1557e16
    for i in range(1, len(z_sorted)):
        dt = t_arr[i] - t_arr[i - 1]
        ndot = ionizingEmissivity(z_sorted[i], f_esc=f_esc, log_xi_ion=log_xi_ion)
        ndot_cgs = ndot / MPC_CM**3
        t_rec = recombinationTimescale(z_sorted[i], C_HII=C_HII)
        dQ = (ndot_cgs / n_H0) * dt - (Q[i - 1] / t_rec) * dt
        Q[i] = np.clip(Q[i - 1] + dQ, 0.0, 1.0)
    Q_out = np.empty_like(Q)
    Q_out[order] = Q
    return Q_out

def thomsonOpticalDepth(z_grid, Q_HII):
    order = np.argsort(z_grid)
    z_sorted = np.asarray(z_grid)[order]
    Q_sorted = np.asarray(Q_HII)[order]
    n_H0 = hydrogenNumberDensity()
    n_e = 1.08 * Q_sorted * n_H0 * (1.0 + z_sorted)**3
    t_arr = cosmo.age(z_sorted) * 3.1557e16
    dt_dz = np.gradient(t_arr, z_sorted)
    return SIGMA_T * C_CMS * np.trapezoid(n_e * np.abs(dt_dz), z_sorted)

def gunnPetersonTau(x_HI, z):
    prefactor = 6.45e5 * (cosmo.Om0 * cosmo.h**2)**(-0.5) * (cosmo.Ob0 * cosmo.h**2 / 0.02)
    return prefactor * np.asarray(x_HI) * ((1.0 + np.asarray(z)) / 7.0)**1.5

# ---- 21-cm cosmology helpers (toy global signal) ----

T_CMB_0 = 2.725  # K

def cmbTemperature(z):
    return T_CMB_0 * (1.0 + np.asarray(z, dtype=float))

def kineticTemperature(z, z_heat=15.0, z_dec=150.0, T_heat_norm=200.0):
    """Toy IGM kinetic temperature: adiabatic at high z, X-ray heated at low z."""
    z = np.asarray(z, dtype=float)
    T_adi = T_CMB_0 * (1.0 + z)**2 / (1.0 + z_dec)
    f_heat = 0.5 * (1.0 - np.tanh((z - z_heat) / 2.0))
    T_h = T_heat_norm * np.sqrt(11.0 / (1.0 + z))
    return (1.0 - f_heat) * T_adi + f_heat * T_h

def spinTemperature(z, z_couple=20.0):
    """Toy spin temperature: T_gamma at high z, T_K once Wouthuysen-Field couples."""
    z = np.asarray(z, dtype=float)
    T_K = kineticTemperature(z)
    T_g = cmbTemperature(z)
    f_coup = 0.5 * (1.0 - np.tanh((z - z_couple) / 2.0))
    return 1.0 / ((1.0 - f_coup) / T_g + f_coup / T_K)

def globalDeltaTb(z, x_HI=None):
    """Global 21-cm differential brightness temperature (mK)."""
    z = np.asarray(z, dtype=float)
    if x_HI is None:
        x_HI = np.ones_like(z)
    pre = 27.0 * (cosmo.Ob0 * cosmo.h**2 / 0.023) \
              * np.sqrt(0.15 / (cosmo.Om0 * cosmo.h**2))
    return pre * x_HI * (1.0 - cmbTemperature(z) / spinTemperature(z)) \
               * np.sqrt((1.0 + z) / 10.0)

# ---- UV LF helpers ----

def schechterMag(M_UV, M_star, phi_star, alpha):
    """Schechter LF in absolute-magnitude form (Mpc^-3 mag^-1)."""
    x = 10.0**(-0.4 * (np.asarray(M_UV) - M_star))
    return 0.4 * np.log(10.0) * phi_star * x**(alpha + 1.0) * np.exp(-x)

def rhoUV_from_LF(M_star, phi_star, alpha, M_lim=-17.0, M_bright=-25.0):
    """Integrate Schechter LF to get rho_UV (erg/s/Hz/Mpc^3)."""
    M = np.linspace(M_bright, M_lim, 2000)
    L = 10.0**((51.6 - M) / 2.5)
    return np.trapezoid(schechterMag(M, M_star, phi_star, alpha) * L, M)

def sfrdFromUV(rho_UV):
    """Kennicutt 1998 (Chabrier IMF): SFRD = 1.15e-28 * rho_UV."""
    return 1.15e-28 * rho_UV

## From Chemistry to the First Light

The past several lectures traced galaxy formation from dark matter halos through gas cooling, star formation, feedback, mergers, and scaling relations. Last lecture gave the observational panorama across cosmic time.

Two threads remained unfinished:
- Where do the heavy elements come from, and how does galactic chemical composition evolve?
- How did the Universe transition from the neutral dark ages to the ionized, transparent state we observe today?

Both are driven by the same underlying engine -- the integrated history of star formation -- but they probe very different physics and observables. Chemical evolution is a local story about what happens inside individual galaxies. Reionization is a global story about the intergalactic medium (IGM).

Today we connect them both back to the Madau-Dickinson SFRD and to everything we have built this semester.

## Part I: Chemical Evolution

Before the first stars, the Universe contained only hydrogen, helium, and traces of lithium from Big Bang nucleosynthesis. Every heavier element was made in stars.

Three dominant nucleosynthesis channels:

| Channel | Progenitor | Delay time | Main products |
| --- | --- | --- | --- |
| Type II / Ib / Ic SNe | $M > 8\,M_\odot$ stars | 3-40 Myr | alpha: O, Mg, Si, Ca |
| Type Ia SNe | WD in binary | 0.1-10 Gyr | iron peak: Fe, Ni |
| AGB stars | $1$-$8\,M_\odot$ stars | $\sim$100 Myr-1 Gyr | C, N, s-process |

The different delay times are critical -- they let us use abundance ratios like $[\alpha/{\rm Fe}]$ as a clock for the formation timescale of a stellar population.

## The Closed Box Model

Define the gas mass fraction $\mu \equiv M_{\rm gas} / (M_{\rm gas} + M_*)$, and use the instantaneous recycling approximation: each stellar generation immediately returns a fraction $R$ of its mass to the ISM, enriched with nucleosynthetic yield $y_Z$.

The closed-box solution is:

$$Z(\mu) = y_Z\,\ln(1/\mu)$$

Metallicity grows logarithmically as gas is consumed.

The corresponding metallicity distribution function (cumulative mass of stars with metallicity below $Z$) is:

$$\frac{M_*(<Z)}{M_*(<Z_{\rm final})} = \frac{1 - e^{-Z/y_Z}}{1 - e^{-Z_{\rm final}/y_Z}}$$

When applied to the solar neighborhood, the closed box predicts far more low-metallicity stars than are observed. This is the G-dwarf problem and signals that real galaxies cannot be closed systems.

## Exercise 1: The Closed Box and the G-Dwarf Problem

1. Plot $Z(\mu)$ from $\mu = 0.99$ down to $\mu = 0.01$ for $y_Z = 0.02$ (roughly solar for a massive-star IMF).
2. Evaluate the fraction of stars with $Z < 0.25\,Z_\odot$ predicted by the closed-box model (take $Z_{\rm final} = Z_\odot$).
3. Compare with the observed solar-neighborhood value of $\approx 0.03$-$0.05$. How much does the closed box overpredict?

In [ ]:
# Exercise 1: Closed box + G-dwarf problem

mu = np.linspace(0.01, 0.99, 200)

# FILL IN: closed-box metallicity
Z_closed = # FILL IN

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

ax1 = axes[0]
ax1.plot(mu, Z_closed / Z_SUN, 'b-', lw=2)
ax1.axhline(1.0, ls='--', color='gray', lw=0.8)
ax1.text(0.05, 1.1, r'$Z_\odot$', fontsize=9, color='gray')
ax1.set_xlabel(r'Gas fraction $\mu$')
ax1.set_ylabel(r'$Z / Z_\odot$')
ax1.set_title('Closed-Box Metallicity')
ax1.invert_xaxis()

# MDF: cumulative fraction of stars with Z < Z_bin
Z_bins = np.linspace(0, Z_SUN, 200)
Z_final = Z_SUN
# FILL IN: cumulative MDF
cdf_closed = # FILL IN

ax2 = axes[1]
ax2.plot(Z_bins / Z_SUN, cdf_closed, 'b-', lw=2, label='Closed box')

# Observed solar-neighborhood G-dwarf MDF (schematic)
Z_obs = np.array([0.02, 0.05, 0.1, 0.25, 0.5, 0.75, 1.0, 1.25]) * Z_SUN
cdf_obs = np.array([0.005, 0.01, 0.02, 0.05, 0.18, 0.55, 0.85, 0.98])
ax2.plot(Z_obs / Z_SUN, cdf_obs, 'ko-', ms=5, label='Observed (schematic)')

ax2.set_xlabel(r'$Z / Z_\odot$')
ax2.set_ylabel(r'Cumulative fraction of stars with $Z < $ axis value')
ax2.legend(fontsize=9)
ax2.set_title('G-Dwarf Problem')

plt.tight_layout()
plt.show()

# FILL IN: fraction of stars with Z < 0.25 Z_sun
Z_cut = 0.25 * Z_SUN
frac_closed = # FILL IN
print(f'Closed-box fraction with Z < 0.25 Z_sun: {frac_closed:.3f}')
print(f'Observed:                               ~ 0.03-0.05')
print(f'Ratio:                                  ~ {frac_closed/0.04:.1f}x too many')

## Demonstration: The Bathtub Model

The closed-box problem is resolved by coupling accretion, star formation, and outflows. In the "bathtub" limit where gas accretion balances consumption, metallicity approaches a steady-state equilibrium:

$$Z_{\rm eq} = \frac{y_Z}{1 + \eta}$$

where $\eta$ is the mass-loading factor (outflow rate / SFR). Because the equilibrium is stable, galaxies spend most of their time near $Z_{\rm eq}$ -- so the metallicity distribution is sharply peaked rather than broadly spread, naturally eliminating the G-dwarf problem.

The plot below compares the closed box to accreting boxes with $\eta = 0.5, 1, 2, 4$. Notice that the equilibrium metallicity decreases as $\eta$ increases -- low-mass galaxies with strong outflows have lower metallicities. This is the physical origin of the mass-metallicity relation.

In [ ]:
# Bathtub model vs closed box

mu = np.linspace(0.01, 0.99, 200)

fig, ax = plt.subplots(figsize=(7, 4.5))

Z_cb = closedBoxZ(mu)
ax.plot(mu, Z_cb / Z_SUN, 'k-', lw=2.5, label='Closed box')

for eta, color in zip([0.5, 1.0, 2.0, 4.0], ['C0', 'C1', 'C2', 'C3']):
    Z_ab = accretingBoxZ(mu, eta=eta)
    ax.plot(mu, Z_ab / Z_SUN, '-', color=color, lw=2,
            label=fr'Bathtub, $\eta={eta}$ ($Z_{{\rm eq}}={Y_Z_DEFAULT/(1+eta)/Z_SUN:.2f}\,Z_\odot$)')

ax.axhline(1.0, ls='--', color='gray', lw=0.8)
ax.set_xlabel(r'Gas fraction $\mu$')
ax.set_ylabel(r'$Z / Z_\odot$')
ax.legend(fontsize=9)
ax.set_title('Closed Box vs Bathtub Model')
ax.invert_xaxis()
plt.tight_layout()
plt.show()

## Demonstration: The Mass-Metallicity Relation

Tremonti et al. (2004) used SDSS to measure gas-phase oxygen abundances for ~53,000 star-forming galaxies, finding a tight correlation:

$$12 + \log({\rm O/H}) \approx 8.69 + 0.5\,\log\left(\frac{M_*}{10^{10}\,M_\odot}\right)$$

for $M_* \lesssim 10^{10.5}\,M_\odot$, saturating at solar ($12 + \log {\rm O/H} \approx 8.7$) at higher masses.

This is explained by mass-dependent outflow efficiency: low-mass galaxies have shallow potential wells, so SN winds eject metal-rich gas efficiently (high $\eta$), keeping the equilibrium metallicity low. Massive galaxies retain more of their metals (low $\eta$), approaching the closed-box yield.

The MZR evolves with redshift -- galaxies at $z \sim 2$ at fixed stellar mass are about 0.3 dex more metal-poor than their local counterparts, reflecting both their earlier evolutionary stage and enhanced pristine gas accretion at cosmic noon.

In [ ]:
# Mass-metallicity relation at multiple redshifts

log_Mstar = np.linspace(8.5, 11.5, 100)

fig, ax = plt.subplots(figsize=(7, 4.5))

for z_val, color in zip([0.0, 0.7, 1.5, 2.5], ['black', 'C0', 'C1', 'C3']):
    log_OH = massMetallicityRelation(log_Mstar, z=z_val)
    ax.plot(log_Mstar, log_OH, color=color, lw=2, label=f'$z = {z_val}$')

ax.axhline(LOGOH_SUN, ls='--', color='gray', lw=0.8)
ax.text(8.6, LOGOH_SUN + 0.02, 'Solar', fontsize=9, color='gray')

ax.set_xlabel(r'$\log_{10}\,M_*\,(M_\odot)$')
ax.set_ylabel(r'$12 + \log_{10}(\mathrm{O/H})$')
ax.legend(fontsize=10)
ax.set_title('Mass-Metallicity Relation vs Redshift')
plt.tight_layout()
plt.show()

## Demonstration: Alpha Enhancement and Downsizing

Elliptical galaxies and massive bulges show $[\alpha/{\rm Fe}] \sim +0.2$ to $+0.4$ -- enhanced alpha abundances relative to iron compared to the Sun. This is the chemical fingerprint of short, intense star formation: alpha elements are deposited quickly by Type II SNe, but iron from Type Ia SNe has not had time to catch up.

A rough empirical relation connects $[\alpha/{\rm Fe}]$ to the characteristic star-formation timescale:

$$[\alpha/{\rm Fe}] \approx 0.3 - 0.3\,\log_{10}(\tau_{\rm SF} / 1\,{\rm Gyr})$$

Massive ellipticals with $[\alpha/{\rm Fe}] = +0.3$ formed in about 0.5 Gyr. Low-mass disk galaxies with $[\alpha/{\rm Fe}] = 0$ formed over 10 Gyr -- the full age of the Universe. This is a direct chemical confirmation of downsizing from Lecture 23.

In [ ]:
# Alpha-enhancement vs star formation timescale

tau = np.logspace(-0.5, 1.3, 100)  # 0.3 to 20 Gyr
alphaFe = alphaFeVsTimescale(tau)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(tau, alphaFe, 'b-', lw=2)

# Annotate galaxy types
ax.plot(0.5, alphaFeVsTimescale(0.5), 'ro', ms=10)
ax.annotate('Massive ellipticals', xy=(0.5, alphaFeVsTimescale(0.5)),
            xytext=(0.7, 0.32), fontsize=10)
ax.plot(3.0, alphaFeVsTimescale(3.0), 'go', ms=10)
ax.annotate('Milky Way bulge', xy=(3.0, alphaFeVsTimescale(3.0)),
            xytext=(4.0, 0.18), fontsize=10)
ax.plot(10.0, alphaFeVsTimescale(10.0), 'co', ms=10)
ax.annotate('Disk galaxies', xy=(10.0, alphaFeVsTimescale(10.0)),
            xytext=(3.0, -0.04), fontsize=10)

ax.axhline(0.0, ls='--', color='gray', lw=0.8)
ax.set_xscale('log')
ax.set_xlabel(r'Star-formation timescale $\tau_{\rm SF}$ (Gyr)')
ax.set_ylabel(r'$[\alpha/\mathrm{Fe}]$')
ax.set_title('Chemical Signature of Downsizing')
plt.tight_layout()
plt.show()

## Part II: Cosmic Reionization

Now we rewind the clock to the opposite end of cosmic history.

After recombination at $z \approx 1100$, the Universe was filled with neutral hydrogen and cosmic microwave background photons. There were no stars and no sources of visible light: the Dark Ages had begun. They lasted several hundred million years.

Small density perturbations eventually collapsed into the first dark matter halos. Gas fell in, cooled (first through molecular hydrogen at $T \sim 200$-$10^4$ K), and formed the first stars -- Population III stars, made from pristine H and He. These massive, hot stars emitted copious UV photons that ionized their surroundings, beginning the process of reionization.

By $z \approx 6$ (1 Gyr after the Big Bang), reionization was essentially complete. Today we see fingerprints of this transition in:
- The Thomson optical depth of the CMB (Planck 2018: $\tau_e = 0.054 \pm 0.007$)
- The Gunn-Peterson trough in high-redshift quasar spectra
- JWST spectra of galaxies in the Epoch of Reionization (EoR)
- The (still-contested) global 21-cm absorption signal

## The Ionizing Photon Budget

Reionization requires producing at least one ionizing photon for every hydrogen atom, plus additional photons to offset recombinations. The balance equation for the volume-averaged ionized fraction $Q_{\rm HII}$ is:

$$\dot{Q}_{\rm HII} = \frac{\dot{n}_{\rm ion}(z)}{\bar{n}_H} - \frac{Q_{\rm HII}}{t_{\rm rec}(z)}$$

The ionizing emissivity from galaxies is

$$\dot{n}_{\rm ion}(z) = f_{\rm esc}\,\xi_{\rm ion}\,\rho_{\rm UV}(z)$$

with three key parameters:
- $f_{\rm esc}$: fraction of ionizing photons that escape the galaxy ($\sim 0.05$-$0.2$)
- $\xi_{\rm ion}$: ionizing photon production efficiency per unit UV luminosity ($\sim 10^{25.3}$ Hz/erg for a Chabrier IMF)
- $\rho_{\rm UV}(z)$: non-ionizing UV luminosity density (set by the SFRD via the Kennicutt conversion)

The recombination timescale is

$$t_{\rm rec} = [\,C_{\rm HII}\,\alpha_B(T)\,n_H\,(1+z)^3\,]^{-1}$$

with $\alpha_B(T \approx 2 \times 10^4\,{\rm K}) \approx 2.6 \times 10^{-13}\,{\rm cm}^3\,{\rm s}^{-1}$ and clumping factor $C_{\rm HII} \sim 3$.

## Exercise 2: Reionization from the Madau-Dickinson SFRD

1. Integrate the reionization balance equation from $z = 20$ down to $z = 4$ using the provided `reionizationHistory` routine, with fiducial values $f_{\rm esc} = 0.2$, $\log\,\xi_{\rm ion} = 25.3$, $C_{\rm HII} = 3$.
2. At what redshift does $Q_{\rm HII}$ reach 0.5? 0.90?
3. Repeat for $f_{\rm esc} = 0.1$ and $f_{\rm esc} = 0.3$ and overplot. How sensitive is the reionization endpoint to $f_{\rm esc}$?

In [ ]:
# Exercise 2: Reionization history from the SFRD

z_grid = np.linspace(0.01, 20.0, 500)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# Left: fiducial ionization history
ax1 = axes[0]
# FILL IN: fiducial f_esc = 0.2
Q_fid = # FILL IN
ax1.plot(z_grid, Q_fid, 'b-', lw=2, label=r'$f_{\rm esc}=0.2$ (fiducial)')
ax1.axhline(0.5, ls=':', color='gray', lw=0.8)
ax1.axhline(0.90, ls=':', color='gray', lw=0.8)
ax1.axvspan(6.0, 7.0, alpha=0.1, color='gold')
ax1.set_xlabel('Redshift $z$')
ax1.set_ylabel(r'$Q_{\rm HII}$ (ionized fraction)')
ax1.set_xlim(4, 15)
ax1.set_ylim(0, 1.05)
ax1.legend(fontsize=10)
ax1.set_title('Reionization History')
ax1.invert_xaxis()

# FILL IN: find z where Q = 0.5 and Q = 0.99 (fiducial)
z_half =  # FILL IN
z_end =   # FILL IN  (Q = 0.90)
print(f'Fiducial: Q=0.5  at z = {z_half:.2f}')
print(f'Fiducial: Q=0.90 at z = {z_end:.2f}')

# Right: f_esc sensitivity
ax2 = axes[1]
for f, color in zip([0.1, 0.2, 0.3], ['C0', 'k', 'C3']):
    Q = reionizationHistory(z_grid, f_esc=f)
    ax2.plot(z_grid, Q, color=color, lw=2, label=fr'$f_{{\rm esc}}={f}$')

ax2.axhline(0.5, ls=':', color='gray', lw=0.8)
ax2.axhline(0.90, ls=':', color='gray', lw=0.8)
ax2.set_xlabel('Redshift $z$')
ax2.set_ylabel(r'$Q_{\rm HII}$')
ax2.set_xlim(4, 15)
ax2.set_ylim(0, 1.05)
ax2.legend(fontsize=10)
ax2.set_title(r'Sensitivity to $f_{\rm esc}$')
ax2.invert_xaxis()

plt.tight_layout()
plt.show()

## The CMB Thomson Optical Depth

Free electrons from reionization scatter CMB photons. The integrated Thomson optical depth is:

$$\tau_e = c\,\sigma_T \int_0^{z_{\rm max}} n_e(z)\,\left|\frac{dt}{dz}\right|\,dz$$

where $n_e(z) = x_e(z)\,\bar{n}_H\,(1+z)^3$ and $x_e \approx 1.08\,Q_{\rm HII}$ (with the 0.08 accounting for singly-ionized helium, which shares hydrogen's ionization history).

Measurement history:
- WMAP (2003-2013): $\tau_e \approx 0.09$-$0.17$, implying reionization as early as $z \approx 11$-$17$
- Planck (2018): $\tau_e = 0.054 \pm 0.007$, implying a midpoint of reionization near $z \approx 7.7$

The Planck downward revision brought reionization into much better agreement with galaxy-driven models. Before Planck, exotic high-redshift sources (Pop III clusters, high-mass X-ray binaries) were needed to produce enough ionizing photons at $z > 10$.

## Exercise 3: Thomson Optical Depth

1. Compute $\tau_e$ from the fiducial ionization history of Exercise 2 and compare to the Planck 2018 measurement of $0.054 \pm 0.007$.
2. Tabulate $\tau_e$ for $f_{\rm esc} \in \{0.1, 0.15, 0.2, 0.25, 0.3\}$.
3. Which values of $f_{\rm esc}$ are consistent with Planck at the $1\sigma$ level?

In [ ]:
# Exercise 3: Thomson optical depth

z_grid = np.linspace(0.01, 20.0, 500)

# FILL IN: Thomson tau for fiducial history
Q_fid = reionizationHistory(z_grid, f_esc=0.2)
tau_fid = # FILL IN

print(f'Fiducial tau_e (f_esc=0.2): {tau_fid:.4f}')
print(f'Planck 2018: 0.054 +/- 0.007\n')

# Table over f_esc
f_esc_list = [0.10, 0.15, 0.20, 0.25, 0.30]
tau_list = []
for f in f_esc_list:
    # FILL IN: tau for each f_esc
    Q = # FILL IN
    tau = # FILL IN
    tau_list.append(tau)
    consistent = 'yes' if abs(tau - 0.054) < 0.007 else 'no'
    print(f'f_esc = {f:.2f}: tau_e = {tau:.4f}   consistent with Planck 1-sigma? {consistent}')

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.plot(f_esc_list, tau_list, 'bo-', lw=2, ms=8)
ax.axhline(0.054, ls='-', color='red', lw=1, label='Planck 2018 central')
ax.axhspan(0.054 - 0.007, 0.054 + 0.007, alpha=0.15, color='red', label='Planck 1-sigma')
ax.set_xlabel(r'$f_{\rm esc}$')
ax.set_ylabel(r'$\tau_e$')
ax.legend(fontsize=9)
ax.set_title('Thomson Optical Depth vs Escape Fraction')
plt.tight_layout()
plt.show()

## Demonstration: The Gunn-Peterson Trough

Gunn & Peterson (1965) showed that even a tiny neutral hydrogen fraction in the IGM produces a very large Lyman-$\alpha$ optical depth:

$$\tau_{\rm GP}(z) \approx 6.5 \times 10^5\,x_{\rm HI}\,\left(\frac{1+z}{7}\right)^{3/2}$$

A neutral fraction $x_{\rm HI} = 10^{-4}$ gives $\tau_{\rm GP} = 60$ -- complete absorption. The Ly-$\alpha$ forest transitions from partial transmission at $z \sim 5$ to a black trough at $z \gtrsim 6$. This confirms that reionization was essentially complete by $z \sim 6$, with $x_{\rm HI} \lesssim 10^{-4}$ in most of the IGM by that epoch.

Becker et al. (2001) first identified this trough in SDSS $z \sim 6$ quasars. JWST is now pushing to even higher redshifts and probing a patchy end to reionization.

In [ ]:
# Gunn-Peterson optical depth vs neutral fraction

x_HI = np.logspace(-5, 0, 100)

fig, ax = plt.subplots(figsize=(7, 4.5))

for z_val, color in zip([5.0, 6.0, 7.0, 8.0], ['C0', 'k', 'C1', 'C3']):
    tau_GP = gunnPetersonTau(x_HI, z_val)
    ax.loglog(x_HI, tau_GP, color=color, lw=2, label=f'$z = {z_val}$')

ax.axhline(1.0, ls='--', color='gray', lw=0.8)
ax.text(1e-5, 1.3, r'$\tau = 1$ (unit absorption)', fontsize=9, color='gray')

ax.set_xlabel(r'Neutral fraction $x_{\rm HI}$')
ax.set_ylabel(r'$\tau_{\rm GP}$ (Ly-$\alpha$ optical depth)')
ax.legend(fontsize=10)
ax.set_title('Gunn-Peterson Optical Depth')
plt.tight_layout()
plt.show()

print('Key result: even x_HI = 1e-4 gives tau_GP ~ 60 at z = 6 -- total absorption.')
print('The Gunn-Peterson trough requires x_HI < 1e-4 for transmission.')

## 21-cm Cosmology and the Cosmic Dawn

The $\tau_e$ and Gunn-Peterson trough probe the *endpoints* of reionization. The single observable that, in principle, traces the entire $z \sim 30 \to 6$ history is the redshifted 21-cm hyperfine line of neutral hydrogen.

The differential brightness temperature against the CMB is

$$
\delta T_b \approx 27\,{\rm mK}\;x_{\rm HI}\;\frac{T_S - T_\gamma}{T_S}\;\sqrt{\frac{1+z}{10}}\;\left(\frac{\Omega_b h^2}{0.023}\right)\sqrt{\frac{0.15}{\Omega_m h^2}}.
$$

Three temperatures govern the signal:

- $T_\gamma = 2.725\,(1+z)\,{\rm K}$ -- the CMB.
- $T_K$ -- kinetic temperature of the IGM. After thermal coupling to the CMB is lost near $z \sim 150$, the gas cools adiabatically as $T_K \propto (1+z)^2$. X-rays from the first BHs and SN remnants reheat it above $T_\gamma$ at $z \sim 10$-$15$.
- $T_S$ -- the spin temperature of the 21-cm transition. Set by the competition between CMB photons (drive $T_S \to T_\gamma$) and Lyman-$\alpha$ photons via the Wouthuysen-Field effect (drive $T_S \to T_K$).

The story unfolds in four episodes:
1. Dark ages ($z \gtrsim 30$): no Ly-$\alpha$ sources, $T_S = T_\gamma$, $\delta T_b = 0$.
2. Cosmic dawn ($z \sim 17$-$25$): first stars couple $T_S \to T_K$; the gas is colder than the CMB, so we see absorption ($\delta T_b < 0$).
3. Heating epoch ($z \sim 10$-$15$): X-rays drive $T_K > T_\gamma$, so $T_S > T_\gamma$ and the line goes into emission.
4. Reionization ($z \lesssim 8$): $x_{\rm HI} \to 0$, signal vanishes.

EDGES (Bowman et al. 2018) claimed a $\sim -500$ mK absorption at $z \sim 17$ -- much deeper than standard $\Lambda$CDM predictions of $\sim -100$ to $-200$ mK. SARAS-3 (2022) refuted the EDGES detection. HERA, MWA, and SKA are now closing in on the global signal and the 21-cm power spectrum, which together would give us the only direct probe of cosmic dawn.

In [ ]:
# 21-cm: spin / kinetic / CMB temperatures vs redshift

z_grid = np.linspace(6, 50, 400)

T_g = cmbTemperature(z_grid)
T_K = kineticTemperature(z_grid)
T_S = spinTemperature(z_grid)

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(z_grid, T_g, 'r-', lw=2, label=r'$T_\gamma$ (CMB)')
ax.plot(z_grid, T_K, 'C0--', lw=2, label=r'$T_K$ (kinetic gas)')
ax.plot(z_grid, T_S, 'k-', lw=2, label=r'$T_S$ (spin)')

ax.axvspan(15, 25, color='gold', alpha=0.10, label='Cosmic dawn')
ax.set_xlabel('Redshift $z$')
ax.set_ylabel('Temperature [K]')
ax.set_yscale('log')
ax.set_xlim(50, 6)
ax.legend(fontsize=9, loc='lower left')
ax.set_title('IGM Temperatures During the Dark Ages and Cosmic Dawn')
plt.tight_layout()
plt.show()

print('At high z: T_S = T_gamma  (no Lyman-alpha sources, signal vanishes).')
print('Cosmic dawn: T_S decouples from T_gamma toward T_K < T_gamma -> absorption.')
print('Heating epoch: X-rays raise T_K above T_gamma -> emission.')

In [ ]:
# Global 21-cm brightness temperature

z_grid = np.linspace(6, 35, 400)

# Toy ionization history: tanh transition centered at z~7
x_HI = 0.5 * (1.0 + np.tanh((z_grid - 7.0) * 1.5))
dTb = globalDeltaTb(z_grid, x_HI=x_HI)

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(z_grid, dTb, 'b-', lw=2, label=r'Standard $\Lambda$CDM (toy)')

# EDGES (Bowman+2018) claim
ax.errorbar([17.2], [-500], yerr=[100], xerr=[2], fmt='rs', ms=8, capsize=4,
            label='EDGES claim (refuted by SARAS-3)')
ax.axhspan(-220, -50, color='gray', alpha=0.10, label='Standard prediction range')

ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel('Redshift $z$')
ax.set_ylabel(r'$\delta T_b$ [mK]')
ax.set_xlim(35, 6)
ax.set_ylim(-650, 60)
ax.legend(fontsize=9, loc='lower right')
ax.set_title('Global 21-cm Brightness Temperature')
plt.tight_layout()
plt.show()

print(f'Toy peak absorption: {dTb.min():.0f} mK at z = {z_grid[np.argmin(dTb)]:.1f}')
print('Standard models: -50 to -220 mK; EDGES claim was -500 mK at z ~ 17.')
print('HERA upper limits already rule out the deepest models; SKA will measure the signal directly.')

## JWST and the First Galaxies

JWST has transformed the high-redshift frontier in just three years of operation. Key findings relevant to reionization:

Spectroscopically confirmed galaxies at $z > 13$. JADES-GS-z14-0 at $z = 14.32$ (and counting) is less than 300 Myr after the Big Bang. These galaxies are detectable because their UV is surprisingly bright.

A UV luminosity function 3-10x higher than pre-launch models. More bright galaxies means more ionizing photons in principle, but tension with the Planck $\tau_e$ constrains how efficiently they reionize.

Direct measurements of $\xi_{\rm ion}$. NIRSpec spectroscopy shows that high-redshift galaxies have elevated ionizing photon production, with $\log\,\xi_{\rm ion} \approx 25.5$-$25.8$ -- higher than the local value of $25.3$. This could compensate for lower $f_{\rm esc}$ and keep the total ionizing budget consistent with Planck.

Super-solar ionization tracers at $z \sim 8$. Nitrogen-bright spectra (e.g., GN-z11) suggest unusual enrichment histories -- possibly rapid cycles of SN-driven outflows and recycled accretion, or contributions from exotic sources like very massive stars or runaway stellar collisions in dense clusters.

Consistency with downsizing chemically. JWST spectra of $z \sim 3$-$6$ quiescent galaxies show enhanced $[\alpha/{\rm Fe}]$, confirming short formation timescales for the most massive systems -- the same pattern we derived chemically in Part I.

## Exercise 4: The JWST UV LF Excess

JWST has measured the rest-frame UV luminosity function out to $z \sim 14$. Pre-launch models (calibrated to HST and Spitzer at $z \le 9$) predicted a steep decline in the comoving SFRD at $z \gtrsim 10$. JWST shows galaxies that are more abundant and more luminous than those models expected.

In this exercise:

1. Plot the Schechter UV LF at $z = 9, 11, 12.5, 14.5$ using the approximate fits from Donnan et al. (2024).
2. Integrate each LF (down to $M_{\rm UV} = -17$) to get $\rho_{\rm UV}$, and convert to a Kennicutt-implied SFRD.
3. Compare to the Madau-Dickinson extrapolation. At which redshift is the excess largest?
4. Discuss: if the SFRD at $z > 10$ were 5x higher than Madau-Dickinson predicts, would Planck $\tau_e$ still allow it? What other parameters in the photon budget could compensate?

In [ ]:
# Exercise 4: The JWST UV LF excess

# Approximate Schechter fits at high z (Donnan et al. 2024)
JWST_LF = {
     9.0: dict(M_star=-21.10, phi_star=10**-3.40, alpha=-2.00),
    11.0: dict(M_star=-20.95, phi_star=10**-3.69, alpha=-2.13),
    12.5: dict(M_star=-20.55, phi_star=10**-3.95, alpha=-2.16),
    14.5: dict(M_star=-20.30, phi_star=10**-4.36, alpha=-2.20),
}

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# (a) Plot the LFs at each redshift
ax1 = axes[0]
M_grid = np.linspace(-23, -16, 200)
for z, params in JWST_LF.items():
    # FILL IN: Schechter LF at this redshift
    phi = # FILL IN
    ax1.plot(M_grid, phi, lw=2, label=f'$z = {z}$')
ax1.set_yscale('log')
ax1.invert_xaxis()
ax1.set_xlabel(r'$M_{\rm UV}$')
ax1.set_ylabel(r'$\phi(M_{\rm UV})\;[{\rm Mpc}^{-3}\,{\rm mag}^{-1}]$')
ax1.legend(fontsize=9)
ax1.set_title('JWST UV LFs (Donnan+2024)')

# (b) Integrate each LF and compare with Madau-Dickinson
ax2 = axes[1]
z_md = np.linspace(0.0, 16.0, 200)
ax2.plot(z_md, sfrdMadau14(z_md), 'k-', lw=2, label='Madau-Dickinson 2014')

print(f'{"z":>5}  {"rho_UV":>12}  {"SFRD_LF":>10}  {"SFRD_MD":>10}  {"excess":>8}')
print('-' * 56)

z_obs, sfrd_obs = [], []
for z, params in JWST_LF.items():
    # FILL IN: integrate the LF, then convert to SFRD
    rho_UV  = # FILL IN
    sfrd_lf = # FILL IN

    sfrd_md = sfrdMadau14(z)
    z_obs.append(z); sfrd_obs.append(sfrd_lf)
    print(f'{z:5.1f}  {rho_UV:12.3e}  {sfrd_lf:10.4f}  {sfrd_md:10.4f}  {sfrd_lf/sfrd_md:7.2f}x')

ax2.plot(z_obs, sfrd_obs, 'ro', ms=9, label='JWST LF integration')
ax2.set_yscale('log')
ax2.set_xlabel('Redshift $z$')
ax2.set_ylabel(r'$\rho_{\rm SFR}\;[M_\odot\,{\rm yr}^{-1}\,{\rm Mpc}^{-3}]$')
ax2.legend(fontsize=9)
ax2.set_title('Cosmic SFRD: Madau-Dickinson vs JWST')

plt.tight_layout()
plt.show()

## A Semester in One Diagram

Pull on any thread of galaxy formation and the cosmic SFRD comes out the other end. The same $\rho_{\rm SFR}(z)$ that we measured from observations:

- Sets the cosmic stellar mass density (Lecture 23: integrate $(1-R)\,\rho_{\rm SFR}$ over time).
- Drives the metal yield (this lecture, Part I: $\dot Z_{\rm prod} \approx y_Z\,\rho_{\rm SFR}$).
- Powers ionizing photon production (Part II: $\dot n_{\rm ion} = f_{\rm esc}\,\xi_{\rm ion}\,\rho_{\rm UV}$, with $\rho_{\rm UV} \propto \rho_{\rm SFR}$).
- And, integrated over redshift, anchors the CMB Thomson optical depth $\tau_e$.

The figure below traces a single Madau-Dickinson SFRD into all four observables. One curve, four windows on cosmic history -- the closing diagram of the semester.

In [ ]:
# A semester in one diagram: SFRD threads through stars, metals, photons, tau_e

z_thread = np.linspace(0.05, 16.0, 400)
order = np.argsort(z_thread)
z_s = z_thread[order]
t_s_yr = cosmo.age(z_s) * 1e9  # Gyr -> yr

# (1) cosmic SFRD
rho_sfr = sfrdMadau14(z_thread)

# (2) stellar mass density: cumulative integral of (1-R) * SFRD over time
R = 0.41
sfrd_s = sfrdMadau14(z_s)
rho_star_t = np.cumsum((1.0 - R) * sfrd_s * np.gradient(t_s_yr))
rho_star = np.empty_like(rho_star_t); rho_star[order] = rho_star_t

# (3) ionizing emissivity from the SFRD
ndot_ion = ionizingEmissivity(z_thread, f_esc=0.2)

# (4) cumulative tau_e from z=0 outward to a given z_max
Q_thread = reionizationHistory(z_thread, f_esc=0.2)
Q_s = Q_thread[order]
n_H0 = hydrogenNumberDensity()
n_e_proper = 1.08 * Q_s * n_H0 * (1.0 + z_s)**3
t_s_sec = cosmo.age(z_s) * 3.1557e16
dt_dz = np.gradient(t_s_sec, z_s)
integrand = n_e_proper * np.abs(dt_dz)
tau_cum_s = np.concatenate([[0.0], np.cumsum(0.5 * (integrand[:-1] + integrand[1:]) * np.diff(z_s))])
tau_cum_s *= SIGMA_T * C_CMS
tau_thread = np.empty_like(tau_cum_s); tau_thread[order] = tau_cum_s

fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)

axes[0, 0].plot(z_thread, rho_sfr, 'b-', lw=2)
axes[0, 0].set_yscale('log')
axes[0, 0].set_ylabel(r'$\rho_{\rm SFR}\;[M_\odot/{\rm yr}/{\rm Mpc}^3]$')
axes[0, 0].set_title('1. Cosmic SFRD (Madau-Dickinson)')

axes[0, 1].plot(z_thread, rho_star, 'g-', lw=2)
axes[0, 1].set_yscale('log')
axes[0, 1].set_ylabel(r'$\rho_*\;[M_\odot/{\rm Mpc}^3]$')
axes[0, 1].set_title('2. Cosmic stellar mass density')

axes[1, 0].plot(z_thread, ndot_ion, 'r-', lw=2)
axes[1, 0].set_yscale('log')
axes[1, 0].set_xlabel('Redshift $z$')
axes[1, 0].set_ylabel(r'$\dot{n}_{\rm ion}\;[{\rm s}^{-1}\,{\rm Mpc}^{-3}]$')
axes[1, 0].set_title(r'3. Ionizing photon emissivity ($f_{\rm esc}=0.2$)')

axes[1, 1].plot(z_thread, tau_thread, 'k-', lw=2)
axes[1, 1].axhline(0.054, ls='--', color='red', lw=1, label='Planck 2018')
axes[1, 1].axhspan(0.047, 0.061, color='red', alpha=0.15)
axes[1, 1].set_xlabel(r'Integration limit $z_{\rm max}$')
axes[1, 1].set_ylabel(r'$\tau_e(<z_{\rm max})$')
axes[1, 1].legend(fontsize=9)
axes[1, 1].set_title(r'4. Cumulative Thomson $\tau_e$')

for ax in axes.flatten():
    ax.invert_xaxis()

plt.tight_layout()
plt.show()

print('One SFRD curve, four windows on cosmic history.')
print('Stellar mass, metals, ionizing photons, and tau_e all follow from the same integrated history.')

## Summary

1. Chemical evolution is driven by three nucleosynthesis channels with distinct delay times: Type II SNe (alpha elements, 3-40 Myr), Type Ia SNe (iron peak, 0.1-10 Gyr), and AGB stars (C, N, s-process). Delay-time differences make $[\alpha/\mathrm{Fe}]$ a clock for star-formation timescales.

2. Closed-box model predicts $Z = y_Z \ln(1/\mu)$ but fails the G-dwarf problem by over-predicting low-metallicity stars by an order of magnitude. The bathtub (accreting box) model resolves it, with equilibrium $Z_{\rm eq} = y_Z / (1+\eta)$.

3. The mass-metallicity relation is set by mass-dependent outflows: shallow potential wells in low-mass galaxies give high $\eta$ and low equilibrium metallicity. The fundamental metallicity relation adds an SFR anticorrelation that reflects recent pristine gas accretion.

4. Alpha enhancement $[\alpha/\mathrm{Fe}] \sim +0.3$ in massive ellipticals implies $\tau_{\rm SF} \lesssim 1$ Gyr, confirming downsizing chemically.

5. Cosmic reionization was driven by ionizing photons from star-forming galaxies. The balance equation relates $\rho_{\rm SFR}$, $f_{\rm esc}$, and $\xi_{\rm ion}$ to the ionization history. With fiducial $f_{\rm esc} = 0.2$ and $\log\,\xi_{\rm ion} = 25.3$, reionization completes near $z \approx 6$.

6. Planck 2018 measures the Thomson optical depth at $\tau_e = 0.054 \pm 0.007$, pinning the midpoint of reionization to $z \approx 7.7$ and favoring galaxy-driven models.

7. The Gunn-Peterson trough confirms reionization ended by $z \approx 6$.

8. The redshifted 21-cm line is the only direct probe of the dark ages and cosmic dawn. The expected global signal is a $\sim -100$ to $-200$ mK absorption feature at $z \sim 17$-$25$. EDGES (2018) claimed a much deeper detection that was refuted by SARAS-3 (2022); HERA, MWA, and SKA are now closing in on the true signal.

9. JWST has pushed the spectroscopic frontier beyond $z = 14$. The UV LF integrates to a SFRD a few times higher than Madau-Dickinson at $z \sim 9$-$11$, with $\xi_{\rm ion}$ also elevated -- pieces that the next decade of $f_{\rm esc}$ and 21-cm measurements will tie together.

With this, we close out the semester: from cosmological perturbations to dark matter halos, from gas cooling and star formation to feedback, from mergers and scaling relations to the cosmic evolution of the galaxy population -- and now the chemistry of the ISM and the first light of the Universe. Good luck with your final projects.